# Web Data Science
## Week 14 - AI APIs

[Brian C. Keegan, Ph.D.](http://brianckeegan.com/)  
[Assistant Professor, Department of Information Science](https://www.colorado.edu/cmci/people/information-science/brian-c-keegan)  
University of Colorado Boulder  

Copyright and distributed under an [MIT License](https://opensource.org/licenses/MIT)

## Goals

* Use Python libraries to interact with AI APIs
* Perform some basic summarization and classification tasks using APIs

In [ ]:
import numpy as np
import pandas as pd
pd.options.display.max_columns = 200

%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sb

import requests, json

from bs4 import BeautifulSoup

from scipy.spatial.distance import pdist, squareform

## Installing OpenAI wrapper

We are going to use the official OpenAI API Python library: https://github.com/openai/openai-python

**You should only need to run the cell below once.**

In [ ]:
! conda install conda-forge::openai --yes

Confirm that the install worked by loading the `openai` library.

In [ ]:
from openai import OpenAI

### Create API keys and connect to OpenAI API

You will need to make a project API key: https://platform.openai.com/settings/

**Very insecure** Enter the API key information below.

In [ ]:
openai_key = 'abc123'

**More secure** Store the API key in a file to load.

In [ ]:
with open('openai_key.json','r') as f:
    openai_key = json.load(f)['key']

Now connect to the API.

In [ ]:
client = OpenAI(
    api_key=openai_key,
)

## Make a simple request

From the [OpenAI docs on Text generation](https://platform.openai.com/docs/guides/text-generation):

> "User messages contain instructions that request a particular type of output from the model. You can think of user messages as the messages you might type in to ChatGPT as an end user."

In [ ]:
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "What are five things to remember about the ethics of web scraping?",
        }
    ],
    model="gpt-4o-mini",
)

The `chat_completion` is a `ChatCompletion` object ([docs](https://platform.openai.com/docs/api-reference/chat/object)) that contains a variety of other objects, including the text of the response as well as infromation about usage. 

In [ ]:
chat_completion

The string content of the response is returned within the `choices` > `message` > `content` attribute tree.

In [ ]:
print(chat_completion.choices[0].message.content)

We could ask the model to generate five different versions.

In [ ]:
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "What are five things to remember about the ethics of web scraping?",
        }
    ],
    model="gpt-4o-mini",
    n=5
)

Compare the first and fifth versions. You can see how the outputs are just variations on similar themes.

In [ ]:
print(chat_completion.choices[0].message.content)

In [ ]:
print(chat_completion.choices[4].message.content)

Different models have different performance, training data, and costs. Get a list of the currently available models.

In [ ]:
model_list = client.models.list().data

model_list

The `created` attributes are UNIX timestamps. To convert to human-readable timestamps:

In [ ]:
pd.Timestamp(
    model_list[0].created, # Get the first model in the list
    unit = 's'
)

There are other parameters you can pass into the API to get different results. 

In [ ]:
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "What are five things to remember about the ethics of web scraping?",
        }
    ],
    model="gpt-4o-mini",
    temperature = 1
)

print(chat_completion.choices[0].message.content)

### Customize roles

Change the agent role to respond in a different way. From the docs:

> "Messages with the system role act as top-level instructions to the model, and typically describe what the model is supposed to do and how it should generally behave and respond."

In [ ]:
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You provide helpful responses in the form of rhyming couplets."
        },
        {
            "role": "user",
            "content": "What are five things to remember about the ethics of web scraping?",
        }
    ],
    model="gpt-4o-mini",
)

print(chat_completion.choices[0].message.content)

### Assistant role

You can simulate a conversation with GPT and then ask it to use those interactions to inform subsequent responses. Pass an "assistant" role in a message sequence to simulate a previous conversation. From the docs:

> "Messages with the assistant role are presumed to have been generated by the model, perhaps in a previous generation request... They can also be used to provide examples to the model for how it should respond to the current request - a technique known as few-shot learning."

In [ ]:
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "I'm a little teapot.",
        },
        {
            "role": "assistant",
            "content": "Short and stout."
        },
        {
            "role": "user",
            "content": "Here is my handle.",
        },
        {
            "role": "assistant",
            "content": "Here is my spout.",
        },
        {
            "role": "user",
            "content": "When I get all steamed up"
        }
    ],
    model="gpt-4o-mini",
)

print(chat_completion.choices[0].message.content)

### Structured output and few-shot prompting

For some tasks, you don't want free text responses but structured responses. You can enforce a structured output ([docs](https://platform.openai.com/docs/guides/structured-outputs)) to a request to get the data back in something like a JSON format.

The University Catalog lists all the course numbers, titles, descriptions, and prerequisites. Let's see how GPT does with parsing this into a more structured format.

First define a data model.

In [ ]:
from pydantic import BaseModel

class INFOCourse(BaseModel):
    number: str
    credits: str
    title: str
    description: str
    grading: str
    prerequisites: list[str]

Retrieve the data and try a basic strategy for turning it into text.

In [ ]:
catalog_raw = requests.get('https://catalog.colorado.edu/courses-a-z/info/').text
catalog_soup = BeautifulSoup(catalog_raw)
catalog_courses = catalog_soup.find_all('div',{'class':'courseblock'})

print(catalog_courses[-10].text)

Set up a GPT prompt that includes some examples.

In [ ]:
system_prompt = '''
Extract content from this course catalog.

The course number follows a pattern like "INFO 1234"

The credits are inside parentheses like "(3)"

The title follow the credits like "Computation in Society"

The description has multiple sentences

The grading basis starts with a string like "Grading Basis:" and has values like "Letter Grade"

The prerequisites start with a string like "Requisites" and have other course numbers like "INFO 1201"
'''

Run it.

In [ ]:
completion = client.beta.chat.completions.parse(
    model="gpt-4o-2024-08-06",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": catalog_courses[-25].text},
    ],
    response_format=INFOCourse,
)

print(completion.choices[0].message.content)

### Fine tuning

Sometimes a general model does not behave the way you want it to. "Fine tuning" ([docs](https://platform.openai.com/docs/guides/fine-tuning)) is the process of giving a mmodel more specific instructions, training data, and example responses so it behaves closer to how you intend.

I am going to fine-tune a model based on a corpus of all the CU Boulder course descriptions (see Appendix for details on data retrieval and cleaning).

First load the training data.

In [ ]:
training_prompts = []
with open('training_prompts.jsonl','r') as f:
    for line in f:
        training_prompts.append(json.loads(line))

You can see these are structured in the format of a system > user > assistant sequence. The user content values are just the name of the major (maybe we should use course titles?). The assistant content valeus are a single course description.

In [ ]:
training_prompts[6586]

Load the training data into your account.

In [ ]:
client.files.create(
  file=open("training_prompts.jsonl", "rb"),
  purpose="fine-tune"
)

Run a fine-tuning job.

In [ ]:
client.fine_tuning.jobs.create(
  training_file="file-37QVuyFmXwWN2z7acGJCav",
  model="gpt-4o-mini-2024-07-18"
)

Check on the status of the fine-tuning.

In [ ]:
client.fine_tuning.jobs.list(limit=10)

## Apply AI to a web data source

Here are four bills from the 2024 legislative sessions:

* [HB24-1061 "Marijuana Industry and Social Equity"](https://leg.colorado.gov/bills/hb24-1061)
* [SB24-076 "Streamline Marijuana Regulation"](https://leg.colorado.gov/bills/sb24-076)
* [HB24-1468 "Candidate Election Deepfake Disclosures"](https://leg.colorado.gov/bills/hb24-1147)
* [SB24-205 "Consumer Protections for Artificial Intelligence"](https://leg.colorado.gov/bills/sb24-205)

Let's retrieve the bill summaries for each of them.

In [ ]:
bills = [
    {
        'name': 'Marijuana Industry and Social Equity',
        'number': 'HB24-1061',
        'url': 'https://leg.colorado.gov/bills/hb24-1061'
    },{
        'name': 'Streamline Marijuana Regulation',
        'number': 'SB-076',
        'url': 'https://leg.colorado.gov/bills/sb24-076'  
    },{
        'name': 'Candidate Election Deepfake Disclosures',
        'number': 'HB24-1147',
        'url': 'https://leg.colorado.gov/bills/hb24-1147'
    },{
        'name': 'Consumer Protections for Artificial Intelligence',
        'number': 'SB24-205',
        'url': 'https://leg.colorado.gov/bills/sb24-205'
    }
]

Add the text of the bill summaries to each of the bill dictionaries.

In [ ]:
for bill in bills:
    
    # Get the page and turn into soup
    raw = requests.get(bill['url']).text
    soup = BeautifulSoup(raw)
    
    # Update the dictionary to include the summary
    bill['summary'] = soup.find('div',{'id':'bill-summary-top'}).text.strip()
    
# Inspect
bills[0]

Use ChatGPT to write a limerick of the each bill summary. Define a `bill_summary_to_limerick` function.

In [ ]:
system_prompt = '''
    The user will pass descriptions of legislation from the Colorado General Assembly. 
    Summarize the description as a limerick.
'''

def bill_summary_to_limerick(summary_text):
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": summary_text
            },
        ]
    )
    
    return response.choices[0].message.content

Test it.

In [ ]:
print(bill_summary_to_limerick(bills[0]['summary']))

Now run it across all the bills' text.

In [ ]:
for bill in bills:
    bill['limerick'] = bill_summary_to_limerick(bill['summary'])

Inspect.

In [ ]:
for bill in bills:
    print(bill['name']+'\n')
    print(bill['limerick']+'\n\n---\n')

### Compare bill similarity

Embeddings are vector representations of strings that encode semantic information about the content of the strings. The vectors themselves are not interpretable by humans, they're the location in a high-dimensional space. But having strings as numeric vectors makes it really easy to perform tasks like evaluate the similarity of text documents.

Among our four bills, there are two related to marijuana and two related to AI. If we use the [OpenAI embeddings API endpoint](https://platform.openai.com/docs/guides/embeddings) we can convert our bill summaries into embeddings and measure the distance between the embeddings. Ideally, the bills about the same topic should be close to each other and far from the other topic.

In [ ]:
embedding_response = client.embeddings.create(
    input = bills[0]['summary'],
    model = 'text-embedding-3-small'
)

The embedding response is a vector of length 1536.

In [ ]:
len(embedding_response.data[0].embedding)

What do the first few values look like?

In [ ]:
embedding_response.data[0].embedding[:5]

Create embeddings for all the bill summaries.

In [ ]:
def make_embeddings(text):
    embedding_response = client.embeddings.create(
        input = text.replace('\n',' '),
        model = 'text-embedding-3-small'
    )
    
    return embedding_response.data[0].embedding

for bill in bills:
    bill['embedding'] = make_embeddings(bill['summary'])

Combine all the bills' embeddings into a numpy array.

In [ ]:
bill_matrix = np.array([bill['embedding'] for bill in bills])
bill_matrix.shape

Measure the distance between the vectors and visualize the distance matrix as a heatmap.

In [ ]:
ax = sb.heatmap(
    data = squareform(1 - pdist(bill_matrix,metric='cosine')),
    cmap = 'coolwarm',
    annot = True,
    vmin = 0,
    vmax = 1,
    square = True,
    cbar_kws = {'label':'Cosine similarity'}
)

labels = [b['name'] for b in bills]

ax.set_xticklabels(labels,rotation=90)
ax.set_yticklabels(labels,rotation=0);

### Sentiment analysis of bills

Given a bill summary, calculate the sentiment of the summary.

In [ ]:
class Bill_Sentiment(BaseModel):
    sentiment: float
    important_words: list[str]

In [ ]:
system_prompt = '''
You are measuring the sentiment of a legislative document.

The partisan value should be from -1 (associated with very liberal and left-wing politics) to 1 (associated with very conservative and right-wing politics)

The important_words should include identify three categories of types of bills this summary belongs to
'''

completion = client.beta.chat.completions.parse(
    model="gpt-4o-2024-08-06",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": bills[3]['summary']},
    ],
    response_format=Bill_Sentiment,
)

print(completion.choices[0].message.content)

## Anthropic claude

Anthropic is another provider of foundation LLMs under the "Claude" brand ([docs](https://docs.anthropic.com/en/docs/initial-setup)).

In [ ]:
! conda install conda-forge::anthropic --yes

Import to make sure it works.

In [ ]:
import anthropic

Register for an API key and store it in a JSON file. Load it here.

In [ ]:
with open('anthropic_key.json','r') as f:
    anthropic_key = json.load(f)['key']

Connect to the client.

In [ ]:
anthropic_client = anthropic.Anthropic(api_key=anthropic_key)

You need to load your account with credits. So this won't work for you until you put something like $5 in your account.

In [ ]:
message = anthropic_client.messages.create(
    model="claude-3-5-haiku-20241022",
    max_tokens=1000,
    system="Write like you are a Klingon captain from Star Trek.",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "What are five things to remember about the ethics of web scraping?"
                }
            ]
        }
    ]
)


The `message` object contains a lot of context and output like the OpenAI.

In [ ]:
message

Access the output with the content attribute.

In [ ]:
print(message.content[0].text)

## Google generative-ai

Google is playing catch-up with developing and deploying generative AI models. It's flagship foundation model is called "Gemini" but there are other models in its ecosystem.

To access Google's AI APIs, we will need to install a library called "google-generativeai" ([docs](https://ai.google.dev/gemini-api/docs/quickstart?lang=python)).

**You only need to run these cells once.**

In [ ]:
! conda install conda-forge::google-generativeai --yes

You might need this "grpcio-tools" if the `import genai` below is broken. Run it just in case! (I [filed a GitHub issue](https://github.com/google-gemini/generative-ai-python/issues/646) to get them to fix it, let's see if they do).

In [ ]:
! conda install anaconda::grpcio-tools --yes

Import to make sure it works.

In [ ]:
import google.generativeai as genai

You will need to log in to a non-CU Google account to access the free tier to create an API key: https://aistudio.google.com/app/apikey

In [ ]:
with open('gemini_key.json','r') as f:
    gemini_key = json.load(f)['key']

Get a list of models.

In [ ]:
list(genai.list_models())[:5]

Make a simple request.

In [ ]:
genai.configure(api_key=gemini_key)
model = genai.GenerativeModel("gemini-1.5-flash")
response = model.generate_content("Explain how AI works as a limerick.")
print(response.text)

## BROKEN: Huggingface transformers

[Huggingface](https://huggingface.co/) is a community of machine learning practioners and researchers that has models that can also be accessed through an API. 

We need to install the "transformers" library ([docs](https://huggingface.co/docs/transformers/en/installation#install-with-conda)).

**You only need to run these cells once.**

In [ ]:
! conda install conda-forge::transformers --yes

In [ ]:
! conda install conda-forge:huggingface_hub --yes

In [ ]:
! conda install conda-forge::pytorch --yes

Import to make sure it works.

In [ ]:
import torch
from transformers import pipeline

You will need to authenticate with the Huggingface API ([docs](https://huggingface.co/docs/transformers.js/guides/private)). Create a token here: https://huggingface.co/settings/tokens

Save the token to a file like "huggingface_key.json" that is a dictionary with a key "key" and load it here.

In [ ]:
with open('huggingface_key.json','r') as f:
    huggingface_key = json.load(f)['key']

In [ ]:
from huggingface_hub import login

login()

In [ ]:
pipe = pipeline(
    "text-generation", 
    "meta-llama/Meta-Llama-3-8B-Instruct", 
    torch_dtype = torch.bfloat16, 
    device_map="auto"
)

In [ ]:
response = pipe(chat, max_new_tokens=512)

print(response[0]['generated_text'][-1]['content'])

## Appendix

### Fine tuning

Retrieve all the course descriptions from the University Catalog: https://catalog.colorado.edu/courses-a-z/

In [ ]:
catalog_raw = requests.get('https://catalog.colorado.edu/courses-a-z/').text
catalog_soup = BeautifulSoup(catalog_raw)
catalog_index = catalog_soup.find('div',{'id':'atozindex'})
catalog_links = catalog_index.find_all('a')

catalog_links = [link for link in catalog_links if link.has_attr('href')]
catalog_links[:10]

Retrieve course catalog for all majors.

In [ ]:
catalog_dict = {}
for link in catalog_links:
    name = link.text
    raw = requests.get('https://catalog.colorado.edu' + link['href']).text
    soup = BeautifulSoup(raw)
    catalog_dict[name] = [course.find('p',{'class':'courseblockdesc'}).text.strip() for course in soup.find_all('div',{'class':'courseblock'})]

Store results.

In [ ]:
with open('cuboulder_catalog.json','w') as f:
    json.dump(catalog_dict,f)

Make a series of prompts using the data.

In [ ]:
training_prompts = []

for major,description_list in catalog_dict.items():
    user_content = "I want to learn about {major}.".format(major=major)

    for description in description_list:
        updated_prompt = {
            "messages": [
                {
                    "role": "system", 
                    "content": "Ralphie is a chatbot that describes university courses."
                }, 
                {
                    "role": "user",
                    "content": user_content
                }, 
                {
                    "role": "assistant", 
                    "content": "Sure, here is a course you might like: " + description
                }
            ]
            }
        
        training_prompts.append(updated_prompt)
        
len(training_prompts)

In [ ]:
training_prompts[0]

In [ ]:
with open('training_prompts.jsonl','w') as f:
    for item in training_prompts:
        f.write(json.dumps(item) + "\n")